# Getting Started / CQL Language Tour
What the language can express, one `%%cql` cell per idea, each followed by a Python cell that runs
it. CQL is not SQL with clinical vocabulary —
it has its own answers for nulls, for dates that only know their own year, for units, for codes, and
for values whose type is not decided until you look at the row. **Getting any of those subtly wrong
produces plausible-but-wrong rates that fail certification silently**, so each section below states
the rule and then shows the engine keeping it.

**Read it without running it.** The markdown above every cell says exactly what that cell should
print. Run All and compare.

Sections 1–8 are **data-free** — no data is read, and every define resolves to CQL's `Unfiltered`
context, so `__subject_id` is the `'*'` seed and the single row is the answer.
Section 9 brings a handful of FHIR resources, and sections 10–11 need them.

**Runs on serverless — no cluster to attach.** Just **Run All**. If `%%cql` is unfamiliar, start with
**Getting Started / Overview**.

In [0]:
%reload_ext cql_databricks.compiler
from cql_databricks.compiler import cql

In [0]:
# Start clean, so Run All gives the same output however often it runs (CLIENT-SESSION-RESET).
from cql_databricks.compiler import cql, reset

reset(spark)


def show(*defines, library=None):
    """Run ``defines`` and display each result, headed by its name. A `%%cql` cell only declares, so each
    one below is followed by a Python cell that runs it: from the session library, or from the library a
    cell with a `library` line declared."""
    query = cql(library=library) if library else cql()
    frames = query.exec(*defines) if len(defines) > 1 else {defines[0]: query.exec(defines[0])}
    for name, frame in frames.items():
        print(f"--- {name} ---")
        display(frame)

## 1. Arithmetic and the numeric tower
The ordinary part, and the baseline for everything after it. CQL distinguishes `/` (decimal
division) from `div` (truncated integer division) and carries decimals at a fixed scale rather than
as floats — a rate is not a floating-point value.

The `%%cql` cell declares the defines; the Python cell after it runs them with `show(...)`, defined
above, which displays each define as its own table, headed by its name.

| define | `__val` |
|---|---|
| `Sum` | `5` |
| `Product` | `42` |
| `Quotient` | `3.14285714` (a decimal, not `3`) |
| `IntegerQuotient` | `3` |
| `Remainder` | `1` |
| `PoweredUp` | `1024` |

In [0]:
%%cql
define "Sum":             2 + 3
define "Product":         6 * 7
define "Quotient":        22 / 7
define "IntegerQuotient": 22 div 7
define "Remainder":       22 mod 7
define "PoweredUp":       2 ^ 10

In [0]:
show("Sum", "Product", "Quotient", "IntegerQuotient", "Remainder", "PoweredUp")

## 2. Null handling — three-valued logic
A CQL boolean is `true | false | null`, and **null propagation is the single most common source of
subtle bugs in a measure**. Two rules do most of the work:

- **An unknown answer is `null`, not a failure.** Divide by zero is a per-row null, not a crashed job
  — one patient's anomaly cannot fail a population run.
- **A null condition is "not true"**, so it takes the `else` branch, and a null `where` predicate
  *excludes* the row. Everything routes through a `cqlTrue(x)` test that is true only when `x` is
  exactly `true` — never Spark's own boolean defaults, which differ.

| define | `__val` | why |
|---|---|---|
| `DivideByZero` | `null` | spec-defined result, not an error |
| `NullPropagates` | `null` | `1 + null` |
| `TrueAndNull` | `null` | could still be either |
| `FalseAndNull` | `false` | already decided — null cannot rescue it |
| `TrueOrNull` | `true` | already decided |
| `NotNull` | `null` | negating unknown is unknown |
| `NullImpliesTrue` | `true` | `implies` is `not A or B` |
| `Coalesced` | `2` | first non-null |
| `IsNullOnNull` | `true` | the explicit test, which is never itself null |
| `NullConditionGoesElse` | `'else-branch'` | null is not-true |

In [0]:
%%cql
define "DivideByZero":          1 / 0
define "NullPropagates":        1 + null
define "TrueAndNull":           true and null
define "FalseAndNull":          false and null
define "TrueOrNull":            true or null
define "NotNull":               not null
define "NullImpliesTrue":       null implies true
define "Coalesced":             Coalesce(null, 2, 3)
define "IsNullOnNull":          IsNull(null)
define "NullConditionGoesElse": if null then 'then-branch' else 'else-branch'

In [0]:
show(
    "DivideByZero",
    "NullPropagates",
    "TrueAndNull",
    "FalseAndNull",
    "TrueOrNull",
    "NotNull",
    "NullImpliesTrue",
    "Coalesced",
    "IsNullOnNull",
    "NullConditionGoesElse",
)

## 3. Equality (`=`) and equivalence (`~`)
Two different operators that a reader coming from SQL will assume are one. They differ **exactly
where it matters most**: around nulls and around the sloppiness of real data.

- **`=` is truth-functional.** If either side is unknown, the answer is unknown — `1 = null` is
  `null`, and so is `null = null`.
- **`~` asks "are these the same value?"** and always answers `true` or `false`. Two nulls are
  equivalent; a value and a null are not. Strings compare case- and whitespace-insensitively.

The two-null cases below are written `null as Integer` because CQL is **statically typed and a bare
`null` carries no type**. In `1 = null` the comparison takes its type from the other side; with null
on both sides there is no other side, and the compiler rejects the expression rather than guess which
`=` you meant. The cast names the type — the value is still null, and the answers are the same
whichever type you name.

This is why `distinct` in a CQL query deduplicates on **equivalence**, not on SQL's `DISTINCT`,
and why code matching is written with `~`: `'Diabetes'` and `'diabetes'` are
the same clinical string, and a value set full of nulls should not silently collapse into unknowns.

| define | `__val` |
|---|---|
| `IntEqual` | `true` |
| `EqualityWithNull` | `null` |
| `EquivalenceWithNull` | `false` |
| `NullEqualsNull` | `null` |
| `NullEquivNull` | `true` |
| `CaseSensitiveEqual` | `false` |
| `CaseInsensitiveEquiv` | `true` |

In [0]:
%%cql
define "IntEqual":             1 = 1
define "EqualityWithNull":     1 = null
define "EquivalenceWithNull":  1 ~ null
define "NullEqualsNull":       (null as Integer) = (null as Integer)
define "NullEquivNull":        (null as Integer) ~ (null as Integer)
define "CaseSensitiveEqual":   'Diabetes' = 'diabetes'
define "CaseInsensitiveEquiv": 'Diabetes' ~ 'diabetes'

In [0]:
show(
    "IntEqual",
    "EqualityWithNull",
    "EquivalenceWithNull",
    "NullEqualsNull",
    "NullEquivNull",
    "CaseSensitiveEqual",
    "CaseInsensitiveEquiv",
)

## 4. Dates carry their own precision
`@2023` is not "the first instant of 2023". It is **the year 2023, and nothing finer is known**. A
CQL date/time carries its precision as data (physically: the length of its ISO string), and a
comparison that the precision cannot decide returns **`null`** — the engine will not invent
precision it does not have.

Spark's own `DATE`/`TIMESTAMP` carry no precision metadata at all, which is why none of this lowers
onto native Spark date operators.

| define | `__val` | why |
|---|---|---|
| `YearOnly` | `2023` | a year-precision value |
| `FullDate` | `2023-06-15` | a day-precision value |
| `SameDay` | `true` | same precision on both sides — decidable |
| `YearVsDay` | `null` | equal as far as the coarse side knows; that is not "equal" |
| `DecidedByYear` | `true` | 2022 < 2023 is settled at year precision, so precision never comes up |
| `NextDay` | `2013-02-01` | `successor of` steps one unit **at the value's own precision** |

`successor of` is also where the tour first calls the engine's runtime: it emits `cql_temporal_step`,
a Unity Catalog Scala function — which is why this whole notebook runs on
serverless with nothing installed on the compute.

In [0]:
%%cql
define "YearOnly":      @2023
define "FullDate":      @2023-06-15
define "SameDay":       @2023-06-15 = @2023-06-15
define "YearVsDay":     @2023 = @2023-06-15
define "DecidedByYear": @2022 < @2023-06-15
define "NextDay":       successor of @2013-01-31

In [0]:
show("YearOnly", "FullDate", "SameDay", "YearVsDay", "DecidedByYear", "NextDay")

## 5. Timing expressions — the interval algebra
Almost every clinical criterion is a timing question: *did this encounter happen during the
measurement period? does this coverage span overlap that one? was the diagnosis before the
procedure?* CQL answers them with first-class intervals and the full set of Allen relations —
`during`, `overlaps`, `before` / `after`, `includes` / `included in`, `meets`, `starts`, `ends`,
`same X as` — all implemented precision-aware in the engine's own library rather than on Spark's
date operators.

Intervals are typed, and each boundary is independently open or closed. An open bound is normalized
to a closed one by discrete successor/predecessor before the relations run, so `Interval(1, 10]`
starts at `2`.

| define | `__val` |
|---|---|
| `Overlaps` | `true` |
| `PointDuring` | `true` |
| `Before` | `true` |
| `Includes` | `true` |
| `Width` | `9` |
| `OpenStart` | `2` |
| `InMeasurementPeriod` | `true` |

In [0]:
%%cql
define "Overlaps":    Interval[1, 5] overlaps Interval[3, 8]
define "PointDuring": 5 during Interval[1, 10]
define "Before":      @2012-01-15 before @2012-03-20
define "Includes":    Interval[1, 10] includes Interval[3, 4]
define "Width":       width of Interval[1, 10]
define "OpenStart":   start of Interval(1, 10]
define "InMeasurementPeriod":
  @2026-06-15 during Interval[@2026-01-01, @2026-12-31]

In [0]:
show("Overlaps", "PointDuring", "Before", "Includes", "Width", "OpenStart", "InMeasurementPeriod")

### 5a. Naming the precision a relation compares at
Section 4's rule and section 5's relations meet here, and the meeting has teeth.

A measurement period is usually written at **day** precision (`Interval[@2026-01-01, @2026-12-31]`),
while a real encounter period starts at a **millisecond** instant (`2026-01-01T00:00:00.000`). Ask
whether one is `during` the other and the bounds tie — equal as far as the coarse side knows — so
the honest answer is `null`, the row fails its `where`, and the denominator quietly comes back empty
with no error at all. That is a real defect this engine shipped once, in a measure run: **71
qualifying encounters, zero rows, no message.**

The fix is in the language, not in the data: **name the precision on the relation**. `during day of`
says "stop caring below the day", and because the named precision belongs to the *comparison* rather
than to either operand, **both sides truncate to it** before comparing.

| define | `__val` | why |
|---|---|---|
| `Unqualified` | `null` | day bound vs millisecond value — undecidable, and silently so |
| `AtDayPrecision` | `true` | both sides truncated to the day |
| `OutsideAtDayPrecision` | `false` | truncation *decides* the tie — it does not blanket-approve |

In [0]:
%%cql
// The same encounter window, asked the same question, twice.
define "Unqualified":
  Interval[@2026-01-01T00:00:00.000, @2026-01-01T01:00:00.000]
    during Interval[@2026-01-01, @2026-12-31]

define "AtDayPrecision":
  Interval[@2026-01-01T00:00:00.000, @2026-01-01T01:00:00.000]
    during day of Interval[@2026-01-01, @2026-12-31]

// The negative control: a window genuinely outside the period stays false.
define "OutsideAtDayPrecision":
  Interval[@2027-01-01T00:00:00.000, @2027-01-02T00:00:00.000]
    during day of Interval[@2026-01-01, @2026-12-31]

In [0]:
show("Unqualified", "AtDayPrecision", "OutsideAtDayPrecision")

## 6. Quantities and UCUM units
A clinical value is a number **and a unit**, and the unit is not decoration: `500 mg` and `1 g` are
comparable, and getting that wrong is a dosing error. CQL quantities carry their UCUM unit, and
comparison and arithmetic convert. Spark has no quantity type, so a quantity is
carried as a `(value, unit)` struct and the conversion is the engine's.

| define | `__val` |
|---|---|
| `Weight` | `{value: 70, unit: "kg"}` |
| `EqualAcrossUnits` | `true` |
| `LessAcrossUnits` | `true` |
| `GreaterAcrossUnits` | `true` |
| `SumAcrossUnits` | `{value: 1.5, unit: "kg"}` — converted into the **left** operand's unit |

(Decimals are carried at a fixed scale, so the displayed values are `70.00000000` and `1.50000000`.)

In [0]:
%%cql
define "Weight":             70 'kg'
define "EqualAcrossUnits":   1 'kg' = 1000 'g'
define "LessAcrossUnits":    500 'mg' < 1 'g'
define "GreaterAcrossUnits": 2 'm' > 150 'cm'
define "SumAcrossUnits":     1 'kg' + 500 'g'

In [0]:
show("Weight", "EqualAcrossUnits", "LessAcrossUnits", "GreaterAcrossUnits", "SumAcrossUnits")

## 7. Lists, conditionals and parameters
CQL lists are **ordered and allow duplicates** — they are sequences, not sets, so `First` and indexed
access mean something and the order is carried as data. A parameter with a `default`
is a constant for the run; it is the same mechanism a measure uses for its measurement period, and
**Getting Started / Overview** showed how to override one from Python.

| define | `__val` |
|---|---|
| `Numbers` | `[1, 2, 3, 4, 5]` — one row, the whole list as an array |
| `ListCount` | `4` |
| `FirstElement` | `10` |
| `Membership` | `true` |
| `Grade` | `"B"` |
| `OverThreshold` | `true` |

In [0]:
%%cql
parameter "Threshold" default 3

define "Numbers":       { 1, 2, 3, 4, 5 }
define "ListCount":     Count({ 1, 2, 3, 4 })
define "FirstElement":  First({ 10, 20, 30 })
define "Membership":    3 in { 1, 2, 3 }
define "Grade":         case when 1 = 2 then 'A' else 'B' end
define "OverThreshold": 4 > "Threshold"

In [0]:
show("Numbers", "ListCount", "FirstElement", "Membership", "Grade", "OverThreshold")

## 8. Polymorphic choice types — the discriminated struct
A `Choice<T1, T2, …>` is a value that is **one of several statically-known types**. It shows up
whenever the branches of a conditional disagree about type, whenever a
`union` mixes list element types — and, most importantly, at the source, because FHIR is full of
`value[x]` elements (section 11).

The physical encoding is a **discriminated struct** that mirrors FHIR's own `value[x]` convention:
one `__choice` tag naming the active member, plus one typed slot per member of which exactly one is
populated. It is deliberately *not* a `VARIANT` — the member types are known from the compiled ELM,
so a closed tagged struct keeps the dispatch static and the schema honest.

`as` projects the named member (null if that is not the active one) and `is` tests the discriminator.

| define | `__val` |
|---|---|
| `HetIfInt` | `{__choice: "Integer", asInteger: 1, asString: null}` |
| `HetIfStr` | `{__choice: "String", asInteger: null, asString: "x"}` |
| `AsMatch` | `1` |
| `AsMismatch` | `null` — asked for the wrong member |
| `IsMatch` | `true` |
| `IsMismatch` | `false` — a test, so never null |

In [0]:
%%cql
define "HetIfInt":   if true  then 1 else 'x'
define "HetIfStr":   if false then 1 else 'x'
define "AsMatch":    (if true then 1 else 'x') as Integer
define "AsMismatch": (if true then 1 else 'x') as String
define "IsMatch":    (if true then 1 else 'x') is Integer
define "IsMismatch": (if true then 1 else 'x') is String

In [0]:
show("HetIfInt", "HetIfStr", "AsMatch", "AsMismatch", "IsMatch", "IsMismatch")

## 9. Bring a handful of FHIR resources
The last two sections need data. The engine reads every `[Resource]` retrieve from a relation that returns
the resource as a `resource` struct, and optionally `patient_id`, the key of the patient the row belongs to.
In a real deployment that relation is a table or view in your environment's `data` schema, over your own governed
tables.
Here it is eleven rows in DataFrames, handed to this notebook's cells with `use_resolvers`: every later
`%%cql` cell reads its data and value sets through the resolvers it names, and nothing else changes.

The resources are plain **FHIR JSON**, one per row, of mixed types: `FhirJson` reads each row's
`resourceType` and parses it against the compiler's FHIR model. That is not cosmetic: an absent
`value[x]` slot has to be genuinely `null` for section 11 to mean anything, and nothing here declares
a schema by hand. The subject key comes from each resource's own `subject` reference.

The cast:

| patient | conditions | observations | encounters |
|---|---|---|---|
| `p1` | `c1` — SNOMED 44054006 | `o1` — a **Quantity**, 120 mm[Hg] · `o2` — a **string**, `"positive"` | `e1` |
| `p2` | none | none | none |
| `p3` | `c2` — SNOMED 195967001 | `o3` — a **Quantity**, 98 mg/dL | `e2` |

…plus a one-row value-set expansion table: the Diabetes value set contains exactly SNOMED
44054006.

**Expected output:** `types: ['Condition', 'Encounter', 'Observation', 'Patient']`.

In [0]:
from cql_databricks.compiler import FhirJson, use_resolvers
from cql_databricks.resolvers import DataFrameDataResolver, DataFrameTerminologyResolver

RESOURCES = [
    '{"resourceType":"Patient","id":"p1","birthDate":"1990-05-01"}',
    '{"resourceType":"Patient","id":"p2","birthDate":"2010-03-15"}',
    '{"resourceType":"Patient","id":"p3","birthDate":"1975-11-20"}',
    '{"resourceType":"Condition","id":"c1","subject":{"reference":"Patient/p1"},'
    '"code":{"coding":[{"system":"http://snomed.info/sct","code":"44054006"}]},"recordedDate":"2026-02-11"}',
    '{"resourceType":"Condition","id":"c2","subject":{"reference":"Patient/p3"},'
    '"code":{"coding":[{"system":"http://snomed.info/sct","code":"195967001"}]},"recordedDate":"2026-04-02"}',
    '{"resourceType":"Observation","id":"o1","subject":{"reference":"Patient/p1"},'
    '"valueQuantity":{"value":120,"unit":"mm[Hg]"}}',
    '{"resourceType":"Observation","id":"o2","subject":{"reference":"Patient/p1"},"valueString":"positive"}',
    '{"resourceType":"Observation","id":"o3","subject":{"reference":"Patient/p3"},'
    '"valueQuantity":{"value":98,"unit":"mg/dL"}}',
    '{"resourceType":"Encounter","id":"e1","subject":{"reference":"Patient/p1"},"status":"finished",'
    '"period":{"start":"2026-03-01T08:00:00.000","end":"2026-03-05T12:00:00.000"}}',
    '{"resourceType":"Encounter","id":"e2","subject":{"reference":"Patient/p3"},"status":"finished",'
    '"period":{"start":"2026-07-19T09:00:00.000","end":"2026-07-19T10:30:00.000"}}',
]

# The value-set expansion: one row per (valueset, system, code).
VALUESETS = spark.createDataFrame([("urn:cql_databricks:vs:diabetes", "http://snomed.info/sct", "44054006")],
                                  "valueset_id STRING, system STRING, code STRING")

# Each type's rows, parsed against the compiler's FHIR model.
fhir = FhirJson(spark.createDataFrame([(r,) for r in RESOURCES], "resource STRING"), column="resource")
types = fhir.found_types(spark)
use_resolvers(data=DataFrameDataResolver({t: fhir.to_df(spark, t) for t in types}, name="tour"),
              terminology=DataFrameTerminologyResolver(VALUESETS, name="tour"))
print("types:", types)

## 10. Terminology filtering
Clinical logic is code-dominated: a measure is mostly a set of questions about whether a resource's
code is in some published value set. CQL writes that as `[Condition: "Diabetes"]` — a retrieve
constrained by value-set membership.

Two things make it work at scale:

- **Value sets are pre-expanded, never called per row.** Expansion belongs to whoever authored
  the set; CQL on Databricks consumes the expansion as an ordinary table and the membership
  test compiles to a broadcast semi-join. There is no terminology service in the hot path, and the
  expansion version is pinned so a rate can be recomputed.
- **Where you write it does not change what you get.** `[Condition: "Diabetes"]` and
  `[Condition] C where C.code in "Diabetes"` are the same question; the compiler pushes the second
  into the retrieve's code filter so both produce the same plan and the same rows.

These are `context Patient` defines, so now there is a real subject — one row per patient, not one
`'*'` row.

**Expected output:** three tables, three rows each.

| `__subject_id` | `HasAnyCondition` | `HasDiabetes` | `HasDiabetesViaWhere` |
|---|---|---|---|
| `p1` | `true` | `true` | `true` |
| `p2` | `false` | `false` | `false` |
| `p3` | `true` | `false` | `false` |

`p3` is the row that proves the filter is doing work: they have a Condition, coded, just not one in
this value set.

`C.code in "Diabetes"` converts a FHIR `CodeableConcept` to a CQL `Concept`, and that conversion lives in
FHIRHelpers, so the library includes it, as every published measure does. CQL never includes it for you.

In [0]:
%%cql
library TerminologyTour version '1.0.0'
using FHIR version '4.0.1'
include FHIRHelpers version '4.0.1' called FHIRHelpers

valueset "Diabetes": 'urn:cql_databricks:vs:diabetes'

context Patient

define "HasAnyCondition":     exists ([Condition])
define "HasDiabetes":         exists ([Condition: "Diabetes"])
define "HasDiabetesViaWhere": exists ([Condition] C where C.code in "Diabetes")

In [0]:
show("HasAnyCondition", "HasDiabetes", "HasDiabetesViaWhere", library="TerminologyTour")

## 11. Polymorphic choice types over FHIR
Section 8's discriminated struct was a language curiosity. Here is why it exists.

### `value[x]` — the choice at the source
An `Observation.value` may be a Quantity, a string, a CodeableConcept, a boolean… and FHIR stores it
in a *differently named element per type*: `valueQuantity`, `valueString`, `valueCodeableConcept`.
CQL papers over that with one name — you write `O.value` — and `is` / `as` dispatch to the right
physical slot.

The dispatch is gated on the **model**, not on the element's name: only a
genuine FHIR `[x]` choice element is projected to a `<path><Type>` slot. Over a monomorphic element
like `Encounter.status`, `as` is an ordinary downcast. A name-only heuristic here reads a slot that
does not exist, gets null, and silently drops rows.

### `Choice<Encounter, Condition>` — whole resources as members
A choice member does not have to be a scalar. `[Encounter] union [Condition]` unions two retrieves
with *different schemas*, and CQL types the result `List<Choice<Encounter, Condition>>`. Each arm
stays a relation and tags its rows with the active member, so the
two resource schemas align into one Choice-tagged relation — no boxing, no exploding, one scan each.

That define is **list-shaped**, so it is the other result shape **Getting Started / Overview** mentioned: one row per
element, with an `__ordinal` column carrying the order.

**Expected output — `HasQuantityValue`** (one row per patient):

| `__subject_id` | `__val` |
|---|---|
| `p1` | `true` |
| `p2` | `false` |
| `p3` | `true` |

**`StringValues`** — one row, `p1` / `"positive"`. `p2` and `p3` have no string-valued observation,
so they contribute no rows at all (an empty list is no rows, not a null row).

**`EncountersOrConditions`** — four rows, each `__val` a discriminated struct, and an `__ordinal`
that puts the Encounter arm before the Condition arm within each patient (display order is whatever
Spark hands back; `__ordinal` is the authority):

| `__subject_id` | `__val` |
|---|---|
| `p1` | `{__choice: "FHIR.Encounter", __resource_type: "Encounter", __resource_id: "e1"}` |
| `p1` | `{__choice: "FHIR.Condition", __resource_type: "Condition", __resource_id: "c1"}` |
| `p3` | `{__choice: "FHIR.Encounter", __resource_type: "Encounter", __resource_id: "e2"}` |
| `p3` | `{__choice: "FHIR.Condition", __resource_type: "Condition", __resource_id: "c2"}` |

Note what the struct kept: the **identity** of each resource and nothing else. Nothing downstream
navigated a field, so nothing else was read — the choice struct is narrowed to what the query
actually demanded.

In [0]:
%%cql
library ChoiceTour version '1.0.0'
using FHIR version '4.0.1'

context Patient

// value[x]: `is` tests the discriminator, `as` projects the matching member.
define "HasQuantityValue":
  exists ([Observation] O where O.value is FHIR.Quantity)

define "StringValues":
  [Observation] O where O.value is FHIR.string return all O.value as FHIR.string

// A Choice whose members are whole resource bodies with different schemas.
define "EncountersOrConditions":
  [Encounter] union [Condition]

In [0]:
show("HasQuantityValue", "StringValues", "EncountersOrConditions", library="ChoiceTour")

---
### Where next
Every rule in this tour exists because a measure depends on it. **Scenarios / Quality Measurement** puts them
together into a measure over FHIR data, and **Getting Started / Overview** ends by opening up what the compiler
produces along the way. Back to **Start Here** for the index.

In [0]:
# Every cell above ran, so the notebook is green. There are no value assertions here by design: the
# narrative states each expected result, and a compile or runtime failure fails its own cell loudly
# rather than being collected into a summary at the end.
import json
dbutils.notebook.exit(json.dumps({"ok": True, "notebook": "getting_started/02_cql_language_tour"}))